# Joins and Aggregations: Physical Plans Under the Hood

Joins and aggregations are the most **compute-intensive** operations in any Spark job. Writing them
correctly at the API level is necessary but not sufficient — you also need to understand what physical
plan Spark chooses and why, so you can intervene when the default is wrong.

---

## The Join Pipeline

Every Spark join goes through four phases:

```
1. PARSE       — Catalyst reads your API call into a Logical Plan
2. ANALYSE     — Resolves column names and types against the catalog
3. OPTIMISE    — Catalyst applies rules (predicate pushdown, constant folding …)
4. PLAN        — Chooses a Physical Strategy: BroadcastHashJoin, SortMergeJoin, etc.
```

The physical strategy choice has enormous performance implications. We'll see how to read the plan
and nudge Spark toward the right one.

---

## Notebook Roadmap

| Section | Topic |
|---------|-------|
| 1 | Setup & sample DataFrames |
| 2 | All six join types with examples |
| 3 | BroadcastHashJoin vs SortMergeJoin (theory) |
| 4 | Forcing a broadcast join; reading `.explain()` |
| 5 | Full aggregation API in one call |
| 6 | Multi-level aggregations, `cube()`, `rollup()` |
| 7 | Join skew and the salting fix |


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("Joins and Aggregations")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# --- ORDERS (large-ish fact table) ---
orders_data = [
    (1001, 1, "Laptop",       1299.99, "2024-01-10"),
    (1002, 2, "Coffee Maker",   89.50, "2024-01-11"),
    (1003, 1, "Headphones",    199.00, "2024-01-12"),
    (1004, 3, "Desk Chair",    450.00, "2024-01-13"),
    (1005, 4, "Blender",        59.99, "2024-01-14"),
    (1006, 5, "Monitor",       349.00, "2024-01-15"),
    (1007, 6, "Keyboard",       79.99, "2024-01-16"),  # customer_id=6 has no matching customer
    (1008, 2, "Laptop",       1299.99, "2024-01-17"),
]

orders_schema = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", IntegerType(), True),
    StructField("product",     StringType(),  True),
    StructField("amount",      DoubleType(),  True),
    StructField("order_date",  StringType(),  True),
])

orders_df = spark.createDataFrame(orders_data, schema=orders_schema)

# --- CUSTOMERS (small dimension table — perfect for broadcast) ---
customers_data = [
    (1, "Alice",   "New York",    "Gold"),
    (2, "Bob",     "Los Angeles", "Silver"),
    (3, "Charlie", "Chicago",     "Gold"),
    (4, "Diana",   "Houston",     "Bronze"),
    (5, "Eve",     "Phoenix",     "Silver"),
    (7, "Grace",   "Austin",      "Bronze"),  # customer_id=7 has no matching order
]

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name",        StringType(),  True),
    StructField("city",        StringType(),  True),
    StructField("tier",        StringType(),  True),
])

customers_df = spark.createDataFrame(customers_data, schema=customers_schema)

print("=== ORDERS ===")
orders_df.show()
print("=== CUSTOMERS ===")
customers_df.show()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/27 09:11:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


=== ORDERS ===


+--------+-----------+------------+-------+----------+
|order_id|customer_id|     product| amount|order_date|
+--------+-----------+------------+-------+----------+
|    1001|          1|      Laptop|1299.99|2024-01-10|
|    1002|          2|Coffee Maker|   89.5|2024-01-11|
|    1003|          1|  Headphones|  199.0|2024-01-12|
|    1004|          3|  Desk Chair|  450.0|2024-01-13|
|    1005|          4|     Blender|  59.99|2024-01-14|
|    1006|          5|     Monitor|  349.0|2024-01-15|
|    1007|          6|    Keyboard|  79.99|2024-01-16|
|    1008|          2|      Laptop|1299.99|2024-01-17|
+--------+-----------+------------+-------+----------+

=== CUSTOMERS ===


+-----------+-------+-----------+------+
|customer_id|   name|       city|  tier|
+-----------+-------+-----------+------+
|          1|  Alice|   New York|  Gold|
|          2|    Bob|Los Angeles|Silver|
|          3|Charlie|    Chicago|  Gold|
|          4|  Diana|    Houston|Bronze|
|          5|    Eve|    Phoenix|Silver|
|          7|  Grace|     Austin|Bronze|
+-----------+-------+-----------+------+



In [2]:
# ============================================================
# SECTION 2: All Join Types
# ============================================================
# Spark supports: inner, left (left_outer), right (right_outer),
# full (full_outer), left_semi, left_anti, cross.

# INNER JOIN — only rows with matching keys on BOTH sides
print("=== INNER JOIN ===")
inner_df = orders_df.join(customers_df, on="customer_id", how="inner")
inner_df.show()

# LEFT JOIN — all rows from orders; null for customer columns where no match
# Use this when orders is the "source of truth" and you want to flag unmatched rows.
print("=== LEFT OUTER JOIN (order_id=1007 has no customer → nulls) ===")
left_df = orders_df.join(customers_df, on="customer_id", how="left")
left_df.show()

# RIGHT JOIN — all rows from customers; null for order columns where no match
print("=== RIGHT OUTER JOIN (customer Grace id=7 has no order → nulls) ===")
right_df = orders_df.join(customers_df, on="customer_id", how="right")
right_df.show()

# FULL OUTER JOIN — all rows from both sides; nulls where no match on either side
print("=== FULL OUTER JOIN ===")
full_df = orders_df.join(customers_df, on="customer_id", how="full")
full_df.show()

# LEFT SEMI JOIN — returns rows from LEFT where a match EXISTS in RIGHT.
# IMPORTANT: no columns from the right side are returned — it's like a filter.
print("=== LEFT SEMI JOIN — orders for customers that exist ===")
semi_df = orders_df.join(customers_df, on="customer_id", how="left_semi")
semi_df.show()

# LEFT ANTI JOIN — opposite of semi: rows from LEFT with NO match in RIGHT.
# Classic use case: find orphan records, detect data quality issues.
print("=== LEFT ANTI JOIN — orders with NO matching customer ===")
anti_df = orders_df.join(customers_df, on="customer_id", how="left_anti")
anti_df.show()

# CROSS JOIN — every row from left × every row from right (cartesian product).
# DANGEROUS on large tables — use only for small dimension × small dimension.
print("=== CROSS JOIN (tiny example: 2 orders × 2 customers) ===")
cross_df = orders_df.limit(2).crossJoin(customers_df.limit(2))
cross_df.show()

=== INNER JOIN ===
+-----------+--------+------------+-------+----------+-------+-----------+------+
|customer_id|order_id|     product| amount|order_date|   name|       city|  tier|
+-----------+--------+------------+-------+----------+-------+-----------+------+
|          1|    1001|      Laptop|1299.99|2024-01-10|  Alice|   New York|  Gold|
|          1|    1003|  Headphones|  199.0|2024-01-12|  Alice|   New York|  Gold|
|          3|    1004|  Desk Chair|  450.0|2024-01-13|Charlie|    Chicago|  Gold|
|          2|    1002|Coffee Maker|   89.5|2024-01-11|    Bob|Los Angeles|Silver|
|          5|    1006|     Monitor|  349.0|2024-01-15|    Eve|    Phoenix|Silver|
|          4|    1005|     Blender|  59.99|2024-01-14|  Diana|    Houston|Bronze|
|          2|    1008|      Laptop|1299.99|2024-01-17|    Bob|Los Angeles|Silver|
+-----------+--------+------------+-------+----------+-------+-----------+------+

=== LEFT OUTER JOIN (order_id=1007 has no customer → nulls) ===
+-----------+-

## BroadcastHashJoin vs SortMergeJoin — When Spark Chooses Each

Spark's physical planner picks a join strategy based on the **estimated size** of each side.
There are three main strategies you'll encounter:

---

### BroadcastHashJoin (BHJ)

```
Driver broadcasts small table to ALL executors
                         ┌──────────┐
              ┌──────────► Executor │  ← hash table in memory
              │          └──────────┘
  Driver ─────┤
  broadcasts  │          ┌──────────┐
  small table └──────────► Executor │  ← hash table in memory
                         └──────────┘
              Large table partitions are streamed — NO shuffle!
```

- **Trigger**: one side's estimated size < `spark.sql.autoBroadcastJoinThreshold` (default **10 MB**).
- **No shuffle** — fastest join strategy available.
- **Risk**: if the "small" table is actually large, each executor needs to hold its entire in-memory
  hash table → OOM. Always set a tight threshold or broadcast explicitly only when you're sure.

---

### SortMergeJoin (SMJ)

```
Step 1 — SHUFFLE both sides on join key → matching keys land on the same partition
Step 2 — SORT each partition by join key (independently, in parallel)
Step 3 — MERGE-SCAN both sorted streams together (linear pass)
```

- **Default** for large-large joins where neither side fits in executor memory.
- Requires **two expensive shuffles** (one per side) — this is typically the dominant cost.
- But it **scales to any size** because it never holds a full side in memory.

---

### Key Config: `spark.sql.autoBroadcastJoinThreshold`

| Value | Effect |
|-------|--------|
| `10485760` (10 MB, default) | Broadcast if estimated size ≤ 10 MB |
| `-1` | Disable auto-broadcast entirely (always SMJ) |
| `104857600` (100 MB) | Broadcast tables up to 100 MB |

Spark's size **estimate** comes from table statistics (run `ANALYZE TABLE … COMPUTE STATISTICS` in
the Hive metastore) or from the physical file size of Parquet/ORC. For in-memory DataFrames created
with `createDataFrame()`, the estimate is often imprecise — use explicit `F.broadcast()` instead.


In [3]:
# ============================================================
# SECTION 4: Forcing a Broadcast Join; Reading .explain()
# ============================================================

# First, let's see what Spark chose WITHOUT a hint (likely SMJ because
# createDataFrame() stats are uncertain).
print("=== PLAN without broadcast hint ===")
orders_df.join(customers_df, on="customer_id", how="inner").explain(mode="formatted")

# Now force a BroadcastHashJoin by wrapping the small side with F.broadcast().
# The hint overrides the autoBroadcastJoinThreshold size check.
broadcast_join_df = orders_df.join(
    F.broadcast(customers_df),   # <-- hint: send customers to all executors
    on="customer_id",
    how="inner"
)

print("=== PLAN with F.broadcast() hint ===")
# Look for 'BroadcastHashJoin' and 'BroadcastExchange' in the output.
# The absence of 'Exchange hashpartitioning' on both sides confirms no shuffle.
broadcast_join_df.explain(mode="formatted")

# Actually execute to confirm results match the inner join above.
print("=== Result of broadcast join ===")
broadcast_join_df.show()

=== PLAN without broadcast hint ===
== Physical Plan ==
AdaptiveSparkPlan (10)
+- Project (9)
   +- SortMergeJoin Inner (8)
      :- Sort (4)
      :  +- Exchange (3)
      :     +- Filter (2)
      :        +- Scan ExistingRDD (1)
      +- Sort (7)
         +- Exchange (6)
            +- Scan ExistingRDD (5)


(1) Scan ExistingRDD
Output [5]: [order_id#0, customer_id#1, product#2, amount#3, order_date#4]
Arguments: [order_id#0, customer_id#1, product#2, amount#3, order_date#4], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [5]: [order_id#0, customer_id#1, product#2, amount#3, order_date#4]
Condition : isnotnull(customer_id#1)

(3) Exchange
Input [5]: [order_id#0, customer_id#1, product#2, amount#3, order_date#4]
Arguments: hashpartitioning(customer_id#1, 8), ENSURE_REQUIREMENTS, [plan_id=861]

(4) Sort
Input [5]: [order_id#0, customer_id#1, product#2, amount#3, order_date#4]
Arguments: [customer_

In [4]:
# ============================================================
# SECTION 5: Full Aggregation API in One .agg() Call
# ============================================================
# .agg() accepts multiple aggregate expressions simultaneously.
# All are computed in a single pass over the data — very efficient.

agg_df = (
    orders_df
    .groupBy("customer_id")
    .agg(
        F.count("order_id").alias("order_count"),           # total rows per group
        F.sum("amount").alias("total_spent"),               # sum of amounts
        F.avg("amount").alias("avg_order"),                 # mean order value
        F.max("amount").alias("max_order"),                 # largest single order
        F.min("amount").alias("min_order"),                 # smallest single order
        F.countDistinct("product").alias("unique_products"),# distinct product count
        F.collect_list("product").alias("product_list"),    # ordered list (may have dups)
        F.collect_set("product").alias("product_set"),      # deduplicated set (order undefined)
        F.round(F.stddev("amount"), 2).alias("stddev_amount"),  # standard deviation
    )
)

# TIP: collect_list / collect_set are memory-heavy — avoid on high-cardinality groups.
agg_df.show(truncate=False)

+-----------+-----------+-----------+---------+---------+---------+---------------+----------------------+----------------------+-------------+
|customer_id|order_count|total_spent|avg_order|max_order|min_order|unique_products|product_list          |product_set           |stddev_amount|
+-----------+-----------+-----------+---------+---------+---------+---------------+----------------------+----------------------+-------------+
|6          |1          |79.99      |79.99    |79.99    |79.99    |1              |[Keyboard]            |[Keyboard]            |NULL         |
|5          |1          |349.0      |349.0    |349.0    |349.0    |1              |[Monitor]             |[Monitor]             |NULL         |
|1          |2          |1498.99    |749.495  |1299.99  |199.0    |2              |[Laptop, Headphones]  |[Laptop, Headphones]  |778.52       |
|3          |1          |450.0      |450.0    |450.0    |450.0    |1              |[Desk Chair]          |[Desk Chair]          |NULL   

In [5]:
# ============================================================
# SECTION 6: Multi-level groupBy, cube(), rollup()
# ============================================================

# --- Multi-column groupBy ---
# Group by both customer_id AND order_date to get daily per-customer totals.
multi_group_df = (
    orders_df
    .groupBy("customer_id", "order_date")
    .agg(F.sum("amount").alias("daily_total"))
    .orderBy("customer_id", "order_date")
)
print("=== Multi-column groupBy ===")
multi_group_df.show()

# Join with customers to get the tier column for cube/rollup demos.
enriched_df = (
    orders_df
    .join(F.broadcast(customers_df), on="customer_id", how="left")
    .select("order_id", "customer_id", "tier", "product", "amount")
)

# --- rollup() ---
# rollup(col1, col2) computes aggregations at multiple levels of a hierarchy:
#   (tier, product) → subtotals per tier-product combination
#   (tier)          → subtotals per tier (product=null means "all products")
#   ()              → grand total (tier=null, product=null)
print("=== rollup(tier, product) — hierarchical subtotals ===")
rollup_df = (
    enriched_df
    .rollup("tier", "product")
    .agg(F.round(F.sum("amount"), 2).alias("total"), F.count("order_id").alias("orders"))
    .orderBy(F.col("tier").asc_nulls_last(), F.col("product").asc_nulls_last())
)
rollup_df.show()

# --- cube() ---
# cube(col1, col2) computes ALL combinations of grouping columns including nulls:
#   (tier, product), (tier, null), (null, product), (null, null)
# cube() = rollup() PLUS the cross-dimension subtotals.
print("=== cube(tier, product) — all-dimension subtotals ===")
cube_df = (
    enriched_df
    .cube("tier", "product")
    .agg(F.round(F.sum("amount"), 2).alias("total"))
    .orderBy(F.col("tier").asc_nulls_last(), F.col("product").asc_nulls_last())
)
cube_df.show(30)

=== Multi-column groupBy ===
+-----------+----------+-----------+
|customer_id|order_date|daily_total|
+-----------+----------+-----------+
|          1|2024-01-10|    1299.99|
|          1|2024-01-12|      199.0|
|          2|2024-01-11|       89.5|
|          2|2024-01-17|    1299.99|
|          3|2024-01-13|      450.0|
|          4|2024-01-14|      59.99|
|          5|2024-01-15|      349.0|
|          6|2024-01-16|      79.99|
+-----------+----------+-----------+

=== rollup(tier, product) — hierarchical subtotals ===
+------+------------+-------+------+
|  tier|     product|  total|orders|
+------+------------+-------+------+
|Bronze|     Blender|  59.99|     1|
|Bronze|        NULL|  59.99|     1|
|  Gold|  Desk Chair|  450.0|     1|
|  Gold|  Headphones|  199.0|     1|
|  Gold|      Laptop|1299.99|     1|
|  Gold|        NULL|1948.99|     3|
|Silver|Coffee Maker|   89.5|     1|
|Silver|      Laptop|1299.99|     1|
|Silver|     Monitor|  349.0|     1|
|Silver|        NULL|1738.4

In [6]:
# ============================================================
# SECTION 7: Join Skew and the Salting Fix
# ============================================================
# SKEW = one partition holds far more data than the others.
# In a SortMergeJoin, the skewed partition takes 10x longer, stalling
# the whole stage at the last 1-2 tasks (the "long tail" problem).
#
# SALTING FIX: artificially expand the join key with a random integer
# (the "salt") so the skewed partition is split across N buckets.
# The right-side rows are replicated N times (once per salt value).

import random

SALT_BUCKETS = 4  # replicate the right side 4 times

# Simulate a skewed orders dataset: customer_id=1 has 80% of all orders.
skewed_orders_data = (
    [(i, 1, f"Product_{i}", float(i * 10)) for i in range(1, 81)]  # 80 rows for cust 1
    + [(i + 80, 2, f"Product_{i}", float(i * 5)) for i in range(1, 11)]  # 10 rows for cust 2
    + [(i + 90, 3, f"Product_{i}", float(i * 7)) for i in range(1, 11)]  # 10 rows for cust 3
)

skewed_orders_df = spark.createDataFrame(
    skewed_orders_data,
    schema=["order_id", "customer_id", "product", "amount"]
)

print("Skew check — rows per customer:")
skewed_orders_df.groupBy("customer_id").count().orderBy("customer_id").show()

# --- WITHOUT salting: customer_id=1 causes one massive partition ---
print("=== Unskewed join (would cause a slow task for customer_id=1) ===")
unskewed_join = skewed_orders_df.join(
    F.broadcast(customers_df), on="customer_id", how="inner"
)
print(f"Row count: {unskewed_join.count()}")

# --- WITH salting ---
# Step 1: Add a random salt 0..N-1 to the LEFT side (the big skewed table).
salted_orders_df = skewed_orders_df.withColumn(
    "salt", (F.rand() * SALT_BUCKETS).cast("int")  # random integer in [0, SALT_BUCKETS)
).withColumn(
    "salted_key",
    F.concat(F.col("customer_id").cast("string"), F.lit("_"), F.col("salt").cast("string"))
)

# Step 2: Explode the RIGHT side so every original row is duplicated N times,
# once for each salt value.
salt_range_df = spark.range(SALT_BUCKETS).toDF("salt_value")
salted_customers_df = (
    customers_df
    .crossJoin(salt_range_df)  # replicate each customer row SALT_BUCKETS times
    .withColumn(
        "salted_key",
        F.concat(F.col("customer_id").cast("string"), F.lit("_"), F.col("salt_value").cast("string"))
    )
)

# Step 3: Join on the composite salted key instead of the original customer_id.
salted_join_df = salted_orders_df.join(
    salted_customers_df,
    on="salted_key",
    how="inner"
).drop("salt", "salt_value", "salted_key")  # clean up salt columns

print("=== After salting — same row count, but partition work is balanced ===")
print(f"Row count: {salted_join_df.count()}")
salted_join_df.show(5)

# NOTE: salting adds SALT_BUCKETS × |customers| rows to the right side — only
# viable when the right side is small (or broadcastable after explosion).
# For large-large skewed joins, use AQE skew join (spark.sql.adaptive.skewJoin.enabled=true).

Skew check — rows per customer:
+-----------+-----+
|customer_id|count|
+-----------+-----+
|          1|   80|
|          2|   10|
|          3|   10|
+-----------+-----+

=== Unskewed join (would cause a slow task for customer_id=1) ===
Row count: 100
=== After salting — same row count, but partition work is balanced ===
Row count: 100
+--------+-----------+----------+------+-----------+-----+--------+----+
|order_id|customer_id|   product|amount|customer_id| name|    city|tier|
+--------+-----------+----------+------+-----------+-----+--------+----+
|      50|          1|Product_50| 500.0|          1|Alice|New York|Gold|
|      47|          1|Product_47| 470.0|          1|Alice|New York|Gold|
|      45|          1|Product_45| 450.0|          1|Alice|New York|Gold|
|      40|          1|Product_40| 400.0|          1|Alice|New York|Gold|
|      31|          1|Product_31| 310.0|          1|Alice|New York|Gold|
+--------+-----------+----------+------+-----------+-----+--------+----+
onl

In [7]:
# Clean shutdown.
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.
